# Data Analysis & Modeling: Predict Weekly Play Time: Zeros and a Long Tail

**Original mock inspired by collected interview topics, not a reported Roblox coding question.** All data are synthetic. Format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. Explain your assumptions and choices while coding.

## Interviewer prompt

At the beginning of day 0, a game platform wants to predict each player's total play time during the next seven days. Some players will play zero minutes; a few may play much more than most others.

**Problem 1 — Data analysis (14 minutes).** Inspect two CSVs, clean the historical sessions, and create one training row per player. Explain missing-duration handling and summarize the target distribution without discarding legitimate high-play-time users.

**Problem 2 — Modeling (21 minutes).** Compare a simple regression model with a model that learns a transformed target. How would you decide which one to use without repeatedly checking the test set? Include a constant baseline, evaluate predictions in minutes, and demonstrate prediction for a new player.

Explain your assumptions and choices while coding. A scikit-learn solution is sufficient; no neural network or custom optimizer is required.

**Familiar:** missing-value checks, Boolean filtering, groupby/agg, left merge, Pipeline, Ridge, MAE and RMSE.

**New:** a three-way split, validation-based model selection, and explicit target transformations with np.log1p/np.expm1. A target transformation is a candidate, not a guaranteed improvement.

| Coding time | Focus |
| --- | --- |
| 0–4 min | Understand the rows and inspect the target |
| 4–8 min | Clean the usable history |
| 8–14 min | Build one feature row per player |
| 14–18 min | Protect a final test set |
| 18–26 min | Compare raw and log-target regression on validation |
| 26–32 min | Freeze the choice and evaluate the held-out test |
| 32–35 min | Predict for a new player and explain the result |

Suggested checkpoints only; they are not official grading criteria. The two problems are data analysis (Steps 1–3) and modeling (Steps 4–7).

**Reference version:** complete code, checks, saved outputs and expandable follow-up answers. Try the practice notebook first.

## Data contract

**Prediction cutoff:** the beginning of day 0. Attributes are snapshots available at that time. Each player appears once. The export includes completed future outcomes for supervised training, but those outcomes cannot be input features.

**players.csv — one row per player**

| Column | Meaning |
| --- | --- |
| user_id | Unique player ID, for joining and checking identity, not a numerical feature |
| account_age_days | Account age at the cutoff; may be missing |
| country, device | Categorical attributes at the cutoff; may be missing |
| next_week_minutes | Target: total play time during days 0–6; nonnegative and observed for every player |
| returned_next_week | A future outcome from days 0–6; exclude it from features |

**sessions.csv — one historical session per row before cleaning**

| Column | Meaning |
| --- | --- |
| session_id | Session ID; exact duplicate export rows may occur |
| user_id | Player ID, always belonging to players.csv |
| day_offset | Day relative to the cutoff; only -7 through -1 inclusive is usable history |
| minutes_played | Duration; a negative sentinel or missing value means unknown, not zero |
| completed | Observed 0/1 completion indicator, valid even when duration is unknown |

**Fixed cleaning and aggregation rules:** Remove exact duplicate session rows, filter to the history window, and turn negative durations into missing values. Keep sessions whose durations are unknown. If an ID still has conflicting rows after exact deduplication, stop and investigate rather than choosing an arbitrary version.

Build session_count (all sessions), known_duration_count (nonmissing durations), total_minutes (sum of observed durations), avg_minutes (mean of observed durations), and completion_rate (mean completed). With sessions but no known durations, total_minutes and avg_minutes stay missing. With no historical sessions, set session_count, known_duration_count, and total_minutes to zero; leave avg_minutes and completion_rate missing. A partial observed total can understate actual history.

Preserve every labeled player. Zero targets are valid. An unusually large nonnegative target is not automatically an error; do not cap or remove authentic high targets merely because they are rare.

For this mock, distinct players at a shared cutoff are independent samples from the same population. Use approximately 60% training, 20% validation, and 20% final test, with SEED = 42. This split evaluates other players from this population, not robustness to a future calendar period. Select by validation MAE in original minutes. Inspect the final test only after the choice is fixed.


## Setup

Use the workspace root as the working directory and select **.venv-codesignal/bin/python**. Inputs are in **data/codesignal_roblox/**. No downloads, data generation, GPU or new dependencies are needed.

Imports and the data path are supplied below.

Do not edit the source CSVs. These exercises deliberately reuse existing files to focus on new decisions rather than a new dataset generator.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn import metrics

SEED = 42
DATA_DIR = Path("data/codesignal_roblox")
assert DATA_DIR.is_dir(), "Open the notebook from the workspace root."


## Problem 1 · Step 1: Understand the rows and inspect the target (0–4 min)

What does one row represent in each file? Load **players** and **sessions**, inspect missingness and suspicious session records, and summarize **next_week_minutes**, including upper percentiles and the fraction equal to zero. Which available columns would be unavailable at prediction time?

**Say before coding:** "I will establish the prediction cutoff, inspect the data contract, and separate legitimate variation from invalid records."

**Possible interviewer questions**

1. Does a high 99th percentile mean most players play that much?
2. Should zero targets or unusually high targets be removed?
3. Why exclude returned_next_week even if it correlates strongly with play time?


In [2]:
players = pd.read_csv(DATA_DIR / "players.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
display(players.head(), sessions.head())
players.info()
display(players.isna().sum(), sessions.isna().sum())
display(players["next_week_minutes"].describe(percentiles=[0.5, 0.9, 0.99]))
print("zero target fraction:", players["next_week_minutes"].eq(0).mean())
print("exact duplicate sessions:", sessions.duplicated().sum())
print("outside history:", (~sessions["day_offset"].between(-7, -1)).sum())
print("negative durations:", sessions["minutes_played"].lt(0).sum())


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes
0,user_0001,184.0,US,mobile,0,0.00
1,user_0002,6.0,US,mobile,0,0.00
2,user_0003,56.0,US,desktop,0,0.00
3,user_0004,39.0,US,desktop,1,247.43
4,user_0005,295.0,GB,NaN,1,5.96


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             1000 non-null   object 
 1   account_age_days    970 non-null    float64
 2   country             950 non-null    object 
 3   device              950 non-null    object 
 4   returned_next_week  1000 non-null   int64  
 5   next_week_minutes   1000 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 47.0+ KB


user_id                0
account_age_days      30
country               50
device                50
returned_next_week     0
next_week_minutes      0
dtype: int64

session_id         0
user_id            0
day_offset         0
minutes_played    99
completed          0
dtype: int64

count    1000.000000
mean       43.316520
std        82.095041
min         0.000000
50%         0.000000
90%       192.697000
99%       290.693100
max       299.290000
Name: next_week_minutes, dtype: float64

zero target fraction: 0.707
exact duplicate sessions: 30
outside history: 30
negative durations: 49


In [3]:
assert players["user_id"].is_unique
assert players["next_week_minutes"].notna().all()
assert players["next_week_minutes"].ge(0).all()
assert sessions["user_id"].isin(players["user_id"]).all()


<details>
<summary>Follow-up answers</summary>

**Does a high 99th percentile mean most players play that much?**

No. It describes an upper-tail boundary: roughly 99% are at or below it. A large gap from the median suggests a relatively small high-play-time tail.

**Should zero targets or unusually high targets be removed?**

Zero is a valid outcome. Large values need domain and logging checks, not automatic removal. Removing authentic high users changes the prediction problem.

**Why exclude returned_next_week even if it correlates strongly with play time?**

It is only known after the future outcome window. Using it would leak information unavailable at the cutoff.

</details>


## Problem 1 · Step 2: Clean the usable history (4–8 min)

Create **clean_sessions** using the fixed validity rules. How will you distinguish an unknown duration from a session that did not happen? Preserve the raw tables and verify that session IDs are unique after exact deduplication.

**Say before coding:** "I will apply deterministic validity rules now and leave learned imputation statistics until after the split."

**Possible interviewer questions**

1. Why not drop every row containing a missing value?
2. Why filter out sessions on day 0 or later?
3. Does drop_duplicates resolve two rows with the same session ID but different durations?


In [4]:
clean_sessions = sessions.drop_duplicates().copy()
assert clean_sessions["session_id"].is_unique, "Investigate conflicting session IDs."
clean_sessions = clean_sessions.loc[
    clean_sessions["day_offset"].between(-7, -1)
].copy()
clean_sessions["minutes_played"] = clean_sessions["minutes_played"].mask(
    clean_sessions["minutes_played"].lt(0)
)


In [5]:
assert clean_sessions["day_offset"].between(-7, -1).all()
assert clean_sessions["minutes_played"].dropna().ge(0).all()
assert clean_sessions["session_id"].is_unique
assert len(clean_sessions) <= len(sessions)


<details>
<summary>Follow-up answers</summary>

**Why not drop every row containing a missing value?**

A session with unknown duration still contributes to session count and completion rate. Dropping it would discard valid information and potentially bias the sample.

**Why filter out sessions on day 0 or later?**

They occur during or after the prediction window begins. Only history available before the cutoff can be used.

**Does drop_duplicates resolve two rows with the same session ID but different durations?**

No. It removes identical rows by default. Conflicting versions require an authoritative resolution rule or investigation; a uniqueness check catches that separate problem.

</details>


## Problem 1 · Step 3: Build one feature row per player (8–14 min)

Create **activity** with the five historical aggregates in the contract, then merge it into players as **features**. Keep players without history. Explain which missing aggregates become zero and which remain unknown. Report the fraction with no history.

**Say before coding:** "I will aggregate before joining so that each label still corresponds to exactly one player row."

**Possible interviewer questions**

1. What is the difference between size and count for session duration?
2. Why correct the sum when every duration is missing?
3. Why retain a player without historical sessions?


In [6]:
activity = clean_sessions.groupby("user_id").agg(
    session_count=("session_id", "size"),
    known_duration_count=("minutes_played", "count"),
    total_minutes=("minutes_played", "sum"),
    avg_minutes=("minutes_played", "mean"),
    completion_rate=("completed", "mean"),
)
activity.loc[activity["known_duration_count"].eq(0), "total_minutes"] = np.nan
features = players.merge(activity, on="user_id", how="left", validate="one_to_one")
no_history = features["session_count"].isna()
features.loc[no_history, ["session_count", "known_duration_count", "total_minutes"]] = 0
print("no-history fraction:", no_history.mean())
display(features.head())


no-history fraction: 0.093


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes,session_count,known_duration_count,total_minutes,avg_minutes,completion_rate
0,user_0001,184.0,US,mobile,0,0.00,0.0,0.0,0.00,NaN,NaN
1,user_0002,6.0,US,mobile,0,0.00,3.0,0.0,NaN,NaN,0.666667
2,user_0003,56.0,US,desktop,0,0.00,1.0,1.0,59.11,59.110000,1.000000
3,user_0004,39.0,US,desktop,1,247.43,4.0,4.0,138.60,34.650000,0.500000
4,user_0005,295.0,GB,NaN,1,5.96,9.0,9.0,342.79,38.087778,0.333333


In [7]:
assert len(features) == len(players)
assert features["user_id"].is_unique
assert features.loc[no_history, "total_minutes"].eq(0).all()
assert features.loc[no_history, ["avg_minutes", "completion_rate"]].isna().all().all()
unknown = features["session_count"].gt(0) & features["known_duration_count"].eq(0)
assert features.loc[unknown, "total_minutes"].isna().all()


<details>
<summary>Follow-up answers</summary>

**What is the difference between size and count for session duration?**

size counts all rows, including sessions with missing duration; count counts nonmissing durations. We need both to distinguish no activity from incomplete measurement.

**Why correct the sum when every duration is missing?**

The default sum can return zero for an all-missing group. That would incorrectly claim no play time was observed as a true zero total.

**Why retain a player without historical sessions?**

They still have a valid future label and may appear at inference. Their absence of history is a useful case, not a reason to remove the player.

</details>


## Problem 2 · Step 4: Protect a final test set (14–18 min)

Which features are available at the cutoff? Create **X** and **y**, then separate approximately 60% training, 20% validation, and 20% final test. Name the six subsets **X_train**, **X_val**, **X_test**, **y_train**, **y_val**, and **y_test**. Explain why two subsets are not enough when choosing between models.

**Say before coding:** "I will use validation to choose the model and reserve the test set for one final estimate after that decision."

**Possible interviewer questions**

1. Why is the second split 25% rather than 20%?
2. Why not choose the transformation based on the best test score?
3. Would this random split answer a future-week deployment question?


In [8]:
numeric_features = ["account_age_days", "session_count", "known_duration_count",
                    "total_minutes", "avg_minutes", "completion_rate"]
categorical_features = ["country", "device"]
X = features[numeric_features + categorical_features]
y = features["next_week_minutes"]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=SEED
)
print("train / validation / test:", len(X_train), len(X_val), len(X_test))


train / validation / test: 600 200 200


In [9]:
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_val) + len(X_test) == len(X)
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert X_test.index.equals(y_test.index)
assert not {"user_id", "returned_next_week", "next_week_minutes"} & set(X.columns)


<details>
<summary>Follow-up answers</summary>

**Why is the second split 25% rather than 20%?**

After holding out 20% for test, 80% remains. Taking 25% of that remainder gives 20% of the original data for validation, leaving 60% for training.

**Why not choose the transformation based on the best test score?**

Then the test set participates in model selection and the reported score is optimistically selected. Validation should make the choice; test should only evaluate the frozen choice.

**Would this random split answer a future-week deployment question?**

Not necessarily. A shared-cutoff random split tests other players in this population. Repeated user snapshots and future-time deployment require an appropriate chronological and possibly grouped split.

</details>


## Problem 2 · Step 5: Compare raw and log-target regression on validation (18–26 min)

Fit a constant baseline and two fresh Ridge models: one learns minutes directly, while the other learns log1p(minutes). Write the target transformation and inverse transformation explicitly: fit the log model on np.log1p(y_train), then apply np.expm1 to its validation predictions to recover minutes. Do not use a target-transform wrapper.

Create **make_ridge**, **raw_ridge**, **log_ridge**, **candidates**, **baseline**, **evaluate**, and a **validation_scores** table. Keep **raw_val_pred** and **log_val_pred** in original minutes. Use MAE in minutes as the primary selection metric; also report RMSE and R2. Fit all preprocessing on training players only. Do not assume the transformed target will win.

**Say before coding:** "I will compare both target choices in the same original units; a log-space loss by itself cannot decide which model best meets a minutes-based objective."

**Possible interviewer questions**

1. Why log1p rather than log?
2. Does inverse-transforming a log prediction recover the conditional mean minutes?
3. Why use a median constant baseline when choosing by MAE?


In [10]:
def make_ridge():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])
    return Pipeline([("preprocess", preprocess), ("regressor", Ridge(alpha=1.0))])


def evaluate(gt, prediction):
    return {
        "MAE_min": metrics.mean_absolute_error(gt, prediction),
        "RMSE_min": metrics.root_mean_squared_error(gt, prediction),
        "R2": metrics.r2_score(gt, prediction),
    }


baseline = DummyRegressor(strategy="median").fit(X_train, y_train)

raw_ridge = make_ridge()
raw_ridge.fit(X_train, y_train)
raw_val_pred = raw_ridge.predict(X_val)

log_ridge = make_ridge()
y_train_log = np.log1p(y_train)
log_ridge.fit(X_train, y_train_log)
log_val_pred = np.expm1(log_ridge.predict(X_val))

candidates = {"raw_ridge": raw_ridge, "log_ridge": log_ridge}
validation_scores = pd.DataFrame({
    "median_baseline": evaluate(y_val, baseline.predict(X_val)),
    "raw_ridge": evaluate(y_val, raw_val_pred),
    "log_ridge": evaluate(y_val, log_val_pred),
}).T
display(validation_scores)


,MAE_min,RMSE_min,R2
median_baseline,36.385050,79.409738,-0.265729
raw_ridge,48.205395,66.581540,0.110183
log_ridge,36.502608,75.491878,-0.143915


In [11]:
assert set(validation_scores.index) == {"median_baseline", "raw_ridge", "log_ridge"}
assert np.isfinite(validation_scores.to_numpy()).all()
raw_imputer = candidates["raw_ridge"].named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"]
np.testing.assert_allclose(raw_imputer.statistics_, X_train[numeric_features].median().to_numpy())
log_pipeline = candidates["log_ridge"]
log_imputer = log_pipeline.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"]
np.testing.assert_allclose(log_imputer.statistics_, X_train[numeric_features].median().to_numpy())
np.testing.assert_allclose(raw_val_pred, candidates["raw_ridge"].predict(X_val))
np.testing.assert_allclose(log_val_pred, np.expm1(candidates["log_ridge"].predict(X_val)))
np.testing.assert_allclose(np.expm1(np.log1p([0.0, 1.0, 100.0])), [0.0, 1.0, 100.0])


<details>
<summary>Follow-up answers</summary>

**Why log1p rather than log?**

log(0) is undefined, while log1p(0) is zero. log1p(y) compresses large positive values without removing valid zero labels. Its inverse is expm1.

**Does inverse-transforming a log prediction recover the conditional mean minutes?**

Not generally: exp(E[log(1+Y) | X]) - 1 is not E[Y | X]. Nonlinearity and model error matter. Treat inverse-transformed predictions as candidates and validate them against the actual business metric.

**Why use a median constant baseline when choosing by MAE?**

The training median minimizes training absolute error among constants; the training mean instead minimizes squared error. Both are legitimate references, but their optimization targets differ.

</details>


## Problem 2 · Step 6: Freeze the choice and evaluate the held-out test (26–32 min)

Choose **best_name** using validation MAE among the two learned models. Create **use_log_target** to record whether the chosen model learns a log target. Refit a fresh **final_model** on the combined training and validation players: pass np.log1p(y_dev) if it is the log model, otherwise pass y_dev. Convert log-model predictions back with np.expm1 before evaluation.

Evaluate once on the held-out test. Refit a **final_baseline** on the same development players and create **test_scores**. What would you conclude if the baseline is competitive?

**Say before coding:** "The choice is fixed before I inspect test labels. I can now refit on all development data, but I will not tune further based on this test result."

**Possible interviewer questions**

1. What if raw Ridge has better RMSE but log-target Ridge has better MAE?
2. What if the chosen model does not beat the constant baseline?
3. Can we clamp negative predictions to zero?


In [12]:
best_name = validation_scores.loc[list(candidates), "MAE_min"].idxmin()
print("selected using validation MAE:", best_name)
use_log_target = best_name == "log_ridge"
final_model = make_ridge()
if use_log_target:
    final_model.fit(X_dev, np.log1p(y_dev))
else:
    final_model.fit(X_dev, y_dev)
final_baseline = DummyRegressor(strategy="median").fit(X_dev, y_dev)
test_pred = final_model.predict(X_test)
if use_log_target:
    test_pred = np.expm1(test_pred)
test_scores = pd.DataFrame({
    "median_baseline": evaluate(y_test, final_baseline.predict(X_test)),
    best_name: evaluate(y_test, test_pred),
}).T
display(test_scores)


selected using validation MAE: log_ridge


,MAE_min,RMSE_min,R2
median_baseline,42.591550,92.200014,-0.271287
log_ridge,42.683134,87.279721,-0.139222


In [13]:
assert best_name in candidates
assert best_name == validation_scores.loc[list(candidates), "MAE_min"].idxmin()
assert use_log_target == (best_name == "log_ridge")
assert set(X_dev.index) == set(X_train.index) | set(X_val.index)
assert set(X_dev.index).isdisjoint(X_test.index)
assert test_pred.shape == y_test.shape
assert np.isfinite(test_pred).all()
assert np.isfinite(test_scores.to_numpy()).all()


<details>
<summary>Follow-up answers</summary>

**What if raw Ridge has better RMSE but log-target Ridge has better MAE?**

That is a real trade-off: RMSE weights large errors more heavily. For this mock, the primary metric was fixed as MAE. If the business instead prioritizes heavy-user error, agree on that objective before selection.

**What if the chosen model does not beat the constant baseline?**

Report it honestly. Check feature signal, target quality, split assumptions, and error segments. Do not claim a successful improvement or switch to another model because of the final test.

**Can we clamp negative predictions to zero?**

That is a plausible serving constraint, but it is a modeling/postprocessing choice. Specify it before selection and apply it consistently in validation and test. This reference reports the original un-clipped predictions.

</details>


## Problem 2 · Step 7: Predict for a new player and explain the result (32–35 min)

Create **new_player**, a single feature row with a missing account age and a previously unseen country. Obtain **new_prediction** using the fitted final model. If use_log_target is true, apply np.expm1 exactly once so the final prediction is in minutes; leave a raw-model prediction unchanged. Explain what preprocessing is reused.

**Say before coding:** "Serving must reuse exactly the fitted preprocessing and the chosen target transformation; prediction should not learn anything from the incoming row."

**Possible interviewer questions**

1. Should the new row be imputed with its own median or fitted again?
2. When should we apply expm1 to final_model.predict?
3. What new situation would make you reconsider this evaluation?


In [14]:
new_player = pd.DataFrame([{
    "account_age_days": np.nan,
    "session_count": 3,
    "known_duration_count": 2,
    "total_minutes": 55.0,
    "avg_minutes": 27.5,
    "completion_rate": 2 / 3,
    "country": "UNSEEN_COUNTRY",
    "device": "mobile",
}])
new_prediction = final_model.predict(new_player)
if use_log_target:
    new_prediction = np.expm1(new_prediction)
print("predicted next-week minutes:", new_prediction[0])


predicted next-week minutes: 1.1692974909784428


In [15]:
assert new_prediction.shape == (1,)
assert np.isfinite(new_prediction).all()
assert set(new_player.columns) == set(X.columns)


<details>
<summary>Follow-up answers</summary>

**Should the new row be imputed with its own median or fitted again?**

No. Reuse the training/development-fitted imputer, scaler and encoder through predict. Fitting on incoming rows changes the learned transformation and breaks consistency.

**When should we apply expm1 to final_model.predict?**

Only when the fitted model learned log1p(minutes). This plain Ridge pipeline returns values in the units of its training target, so the log model needs one explicit inverse transformation. Raw Ridge already predicts minutes. Never apply expm1 twice.

**What new situation would make you reconsider this evaluation?**

A future-week shift, changed logging, or repeated user snapshots can invalidate the shared-cutoff assumptions. Redesign the split and monitor relevant segments rather than assuming this random holdout guarantees future performance.

</details>


## Optional follow-ups — outside the 35-minute core

Optional discussion after the timed mock: a single overall metric can hide errors for heavy users. Using validation predictions only, compare absolute errors for zero, ordinary-positive, and high-play-time targets. Define the high-play-time cutoff using training labels, not the test set. Why might a model with the lowest overall MAE still be unsuitable if heavy-user underprediction is expensive? Do not revise the already reported final test result after this exploration.


In [16]:
high_cutoff = y_train.quantile(0.9)
validation_pred = candidates[best_name].predict(X_val)
if best_name == "log_ridge":
    validation_pred = np.expm1(validation_pred)
validation_errors = pd.DataFrame({
    "actual": y_val,
    "predicted": validation_pred,
})
validation_errors["segment"] = np.select(
    [validation_errors["actual"].eq(0), validation_errors["actual"].gt(high_cutoff)],
    ["zero", "high"], default="positive_not_high",
)
validation_errors["absolute_error"] = (
    validation_errors["actual"] - validation_errors["predicted"]
).abs()
display(validation_errors.groupby("segment").agg(
    players=("actual", "size"),
    MAE_min=("absolute_error", "mean"),
))


,players,MAE_min
segment,,
high,10,239.889886
positive_not_high,49,89.103619
zero,141,3.798194


## Scope and sources

The user-supplied **Data Analysis & Modeling Coding Interview** guide supplies the format: data loading, analysis, feature engineering, a baseline model, evaluation and live explanation in a CodeSignal notebook. It does not specify these exact tasks or APIs.

The local interview collection separates reported **Question/Asked** topics from our explanatory **Answer** material. This mock adapts the topics for practice; it is not evidence that Roblox used this coding problem. Synthetic rates and thresholds are not real product statistics or policy.

This original practice task is inspired by collected report topics: 1161498 (feature construction and evaluation), 1124594 (classification/regression fundamentals), and 995525 (play-time objectives; a more senior interview). The exact CSV task, long-tail comparison, target transformation, and three-way split are teaching adaptations, not a claimed verbatim Roblox question.
